# Spin Qubit Simulator — Demo

Interactive walkthrough of a **1×3 quantum dot array** with 3 electrons,
5 gate electrodes, and qubit frequencies near 5 GHz with 50 MHz splittings.

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so we can import spin_sim directly
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from spin_sim import QuantumDotArray, NoiseModel
from spin_sim.experiments import charge_stability, rabi, ramsey, rb
from spin_sim.plotting import (
    plot_charge_stability, plot_rabi, plot_ramsey, plot_rb
)

## 1. Create the device

In [ ]:
device = QuantumDotArray.default_1x3()
print(device)

## 2. Charge stability diagrams

Select any two of the 5 gates and sweep them to visualise the charge landscape.
Each coloured region corresponds to a charge configuration $(n_1, n_2, n_3)$.

In [ ]:
# Sweep P1 vs P2 (two plunger gates)
csd = charge_stability.run(
    device,
    gate_x="P1", gate_y="P2",
    v_range_x=(-0.05, 0.35),
    v_range_y=(-0.05, 0.35),
    n_points=150,
)
plot_charge_stability(csd);

In [ ]:
# Sweep P1 vs B12 (plunger vs barrier)
csd2 = charge_stability.run(
    device,
    gate_x="P1", gate_y="B12",
    v_range_x=(-0.05, 0.35),
    v_range_y=(0.0, 0.5),
    n_points=150,
)
plot_charge_stability(csd2);

In [ ]:
# Sweep P2 vs P3
csd3 = charge_stability.run(
    device,
    gate_x="P2", gate_y="P3",
    v_range_x=(-0.05, 0.35),
    v_range_y=(-0.05, 0.35),
    n_points=150,
)
plot_charge_stability(csd3);

## 3. Rabi oscillations

Drive qubit 1 (centre dot, $f = 5.000$ GHz) on resonance with a 10 MHz Rabi rate.

In [ ]:
rabi_result = rabi.run(
    device,
    qubit=1,
    drive_frequency_ghz=5.000,
    drive_amplitude_mhz=10.0,
    durations_ns=np.linspace(0, 500, 120),
    shot_count=1024,
    seed=42,
)
plot_rabi(rabi_result);
print(f"Fit: Ω_R = {rabi_result.fit_frequency_mhz:.2f} MHz, π-time = {rabi_result.fit_pi_time_ns:.1f} ns")

### Off-resonance: drive at qubit 1's frequency, observe qubit 0

Qubit 0 is 50 MHz detuned — the drive barely affects it.

In [ ]:
rabi_q0 = rabi.run(
    device,
    qubit=0,
    drive_frequency_ghz=5.000,  # Q1's frequency
    drive_amplitude_mhz=10.0,
    durations_ns=np.linspace(0, 500, 120),
    seed=42,
)
plot_rabi(rabi_result, rabi_q0, labels=["Q1 (on-res)", "Q0 (50 MHz detuned)"]);

### Vary the Rabi rate

In [ ]:
results_sweep = []
for amp in [5.0, 10.0, 20.0]:
    r = rabi.run(
        device, qubit=1, drive_frequency_ghz=5.0,
        drive_amplitude_mhz=amp,
        durations_ns=np.linspace(0, 500, 120), seed=42,
    )
    results_sweep.append(r)

plot_rabi(*results_sweep, labels=["5 MHz", "10 MHz", "20 MHz"]);

## 4. Ramsey fringes

$\pi/2 \to \tau \to \pi/2$ protocol with a 2 MHz detuning. The oscillation frequency
gives the detuning and the envelope gives $T_2^*$.

In [ ]:
ramsey_result = ramsey.run(
    device,
    qubit=1,
    drive_frequency_ghz=5.002,  # 2 MHz detuning
    delays_ns=np.linspace(0, 5000, 200),
    shot_count=1024,
    seed=42,
)
plot_ramsey(ramsey_result);
print(f"Fit: Δ = {ramsey_result.fit_detuning_mhz:.2f} MHz, T₂* = {ramsey_result.fit_t2_star_us:.1f} μs")

### Compare two qubits at different detunings

In [ ]:
ramsey_q0 = ramsey.run(
    device, qubit=0, drive_frequency_ghz=4.955,  # 5 MHz detuning
    delays_ns=np.linspace(0, 3000, 200), seed=42,
)
ramsey_q2 = ramsey.run(
    device, qubit=2, drive_frequency_ghz=5.051,  # 1 MHz detuning
    delays_ns=np.linspace(0, 5000, 200), seed=42,
)
plot_ramsey(ramsey_q0, ramsey_q2, labels=["Q0 (Δ≈5 MHz)", "Q2 (Δ≈1 MHz)"]);

## 5. Randomized benchmarking

Single-qubit Clifford RB to extract gate fidelity.

In [ ]:
rb_result = rb.run(
    device,
    qubit=1,
    clifford_depths=[1, 2, 5, 10, 20, 50, 100, 200, 500],
    sequence_count=30,
    shot_count=1024,
    seed=42,
)
plot_rb(rb_result);
if rb_result.fit_fidelity is not None:
    print(f"Gate fidelity: {rb_result.fit_fidelity*100:.3f}%")
    print(f"EPG: {rb_result.fit_epg:.2e}")

### Compare baseline vs noisy device

In [ ]:
device_noisy = device.with_noise(NoiseModel.noisy())
print("Noisy device:", device_noisy.noise)

rb_noisy = rb.run(
    device_noisy, qubit=1,
    clifford_depths=[1, 2, 5, 10, 20, 50, 100, 200],
    sequence_count=30, shot_count=1024, seed=42,
)

plot_rb(rb_result, rb_noisy, labels=["Baseline", "Noisy"]);

## 6. Ideal device (sanity check)

With zero noise and zero gate error, RB should stay flat at 1.0.

In [ ]:
device_ideal = device.with_noise(NoiseModel.ideal())

rb_ideal = rb.run(
    device_ideal, qubit=1,
    clifford_depths=[1, 5, 10, 50, 100, 500],
    sequence_count=20, shot_count=512, seed=42,
)
plot_rb(rb_ideal, labels=["Ideal"]);